In [ ]:
!pip install -U transformers datasets peft trl bitsandbytes accelerate

In [1]:
import torch
import zipfile

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from datasets import load_dataset
from trl import DPOTrainer, DPOConfig

In [2]:
baseModelName = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(baseModelName)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
zip_path = "/content/Qwen-instructional.zip"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall("/content/")

## DPO (Direct Prefernce Optimization)
1) Base Model (Already SFT trained)
2) Reference Model (Frozen copy of base)
3) Dataset (prompt, chosen, rejected)

In [ ]:
# Base model
base_model = AutoModelForCausalLM.from_pretrained(
    baseModelName,
    device_map="auto",
    torch_dtype=torch.float32
)

# Attach LoRA (SFT model)
model = PeftModel.from_pretrained(
    base_model,
    "/content/checkpoint-12"
)

# Reference model (NO LoRA)
ref_model = AutoModelForCausalLM.from_pretrained(
    baseModelName,
    device_map="auto",
    torch_dtype=torch.float32
)

In [6]:
from google.colab import files
uploaded = files.upload()

dataset = load_dataset("csv", data_files="/content/pharma_preference_data.csv")["train"]

Saving pharma_preference_data.csv to pharma_preference_data (2).csv


Generating train split: 0 examples [00:00, ? examples/s]

In [7]:
dataset[0]

{'prompt': 'Explain the mechanism of action of Metformin in simple scientific terms.',
 'chosen': 'Metformin activates AMPK, which helps cells use glucose more efficiently, reduces glucose production by the liver, and improves insulin sensitivity without increasing insulin levels.',
 'rejected': 'Metformin only lowers blood sugar by stopping sugar absorption from food and does not affect other organs.'}

In [8]:
def format(example):
    return {
        "prompt": example["prompt"],
        "chosen": example["chosen"],
        "rejected": example["rejected"]
    }

dataset = dataset.map(format)

Map:   0%|          | 0/5 [00:00<?, ? examples/s]

In [9]:
for i in range(3):
    print(dataset[i])

{'prompt': 'Explain the mechanism of action of Metformin in simple scientific terms.', 'chosen': 'Metformin activates AMPK, which helps cells use glucose more efficiently, reduces glucose production by the liver, and improves insulin sensitivity without increasing insulin levels.', 'rejected': 'Metformin only lowers blood sugar by stopping sugar absorption from food and does not affect other organs.'}
{'prompt': 'Compare the lipid-lowering effects of Atorvastatin and Ezetimibe when used together versus alone.', 'chosen': 'When used together, Atorvastatin and Ezetimibe provide an additive effect — Atorvastatin reduces cholesterol synthesis in the liver, while Ezetimibe blocks intestinal cholesterol absorption, leading to a greater LDL reduction than either drug alone.', 'rejected': 'Both drugs work the same way, so using them together doesn’t provide any additional cholesterol reduction.'}
{'prompt': 'Summarize the recent developments in mRNA vaccine technology for emerging variants.', 

In [10]:
dpo_config = DPOConfig(
    output_dir="/content/dpo_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=100,
    fp16=False,
    beta=0.1,
    report_to="none"
)

In [ ]:
trainer = DPOTrainer(
    model=model,
    ref_model=ref_model,
    args=dpo_config,
    train_dataset=dataset,
    processing_class=tokenizer
)

In [12]:
torch.cuda.empty_cache()
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss


TrainOutput(global_step=1, training_loss=0.7373381853103638, metrics={'train_runtime': 3.5102, 'train_samples_per_second': 1.424, 'train_steps_per_second': 0.285, 'total_flos': 1260270764544.0, 'train_loss': 0.7373381853103638})

In [13]:
trainer.save_model("/content/dpo_alligned_model")

In [ ]:
prompt = "Explain the mechanism of action of Metformin."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=150,
    temperature=0.7,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

print("\nDPO Model Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))